# SECCIÓN 1: OBJETIVOS Y CONFIGURACIÓN DEL ENTORNO

Este documento constituye el entorno de laboratorio analítico correspondiente a la Semana 6. El propósito analítico es transicionar del enfoque univariado a la estimación paramétrica bivariada.

## Resultados de Aprendizaje Esperados (RAE)
1. **Comprender:** La naturaleza y la geometría espacial de las relaciones bivariadas métricas.
2. **Analizar:** Cuantificar la dependencia lineal empleando la matriz de correlación de Pearson.
3. **Aplicar:** Ajustar e interpretar un estimador de Mínimos Cuadrados Ordinarios (OLS).
4. **Evaluar:** Diagnosticar la calidad del modelo evaluando el Coeficiente de Determinación ($R^2$), la Raíz del Error Cuadrático Medio ($RMSE$) y la homocedasticidad residual.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from typing import Tuple

# Configuración del entorno gráfico (Estilo Académico Sobrio)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelcolor'] = '#003865'
plt.rcParams['axes.titlecolor'] = '#003865'


# SECCIÓN 2: GENERACIÓN SINTÉTICA DEL DATASET OFICIAL

Implementaremos una simulación de Montecarlo estática para modelar el rendimiento académico ($Y$) condicionado a las horas de estudio ($X$). El sistema estructural sigue un proceso determinista subyacente perturbado por un ruido estocástico gaussiano.

Ecuación Estructural Teórica:
$$Y = \beta_0 + \beta_1 X + \epsilon \quad \text{donde } \epsilon \sim \mathcal{N}(0, 15^2)$$


In [ ]:
# Fijar la semilla de entropía para reproducibilidad estricta
np.random.seed(42)

# Definición de parámetros
N = 200
X = np.random.uniform(low=0.0, high=30.0, size=N) # Horas de estudio semanales

# Proceso generador de datos: Y = 20 + 2.5(X) + Ruido
error = np.random.normal(loc=0.0, scale=15.0, size=N)
Y = 20.0 + (2.5 * X) + error

# Inyección controlada de 3 Outliers Extremos para análisis de perturbaciones
X[-3:] = [28.0, 29.5, 30.0]
Y[-3:] = [15.0, 10.0, 5.0] # Estudiantes que estudian mucho pero reprueban drásticamente

Y = np.clip(Y, 0, 100) # Límite lógico de calificación

# Estructura tabular
df = pd.DataFrame({
    'Horas_Estudio': X,
    'Calificacion': Y
})

display(df.describe())


# SECCIÓN 3: EXPLORACIÓN GRÁFICA BIVARIADA (SCATTER PLOTS)

El análisis exploratorio visual resulta imprescindible antes de intentar ajustar estimadores paramétricos, permitiendo identificar distribuciones anómalas (como las ilustradas en el Cuarteto de Anscombe) y violaciones evidentes de supuestos.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, 
    x='Horas_Estudio', 
    y='Calificacion', 
    color='#38BDF8', 
    edgecolor='#003865', 
    s=60
)
plt.title('Diagrama de Dispersión: Relación Bivariada Base', weight='bold')
plt.xlabel('Variable Independiente: Horas de Estudio (X)')
plt.ylabel('Variable Dependiente: Calificación (Y)')
plt.show()


# SECCIÓN 4: MATRIZ DE CORRELACIÓN Y HEATMAPS DIVERGENTES

La correlación de Pearson estandariza la varianza conjunta (Covarianza) para obtener una métrica delimitada en el intervalo $[-1, 1]$. Esta métrica cuantifica la magnitud estricta de la relación lineal.


In [ ]:
matriz_corr = df.corr(method='pearson')

plt.figure(figsize=(6, 5))
sns.heatmap(
    matriz_corr, 
    annot=True, 
    fmt=".3f", 
    cmap='vlag', 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5
)
plt.title('Matriz de Correlación (Pearson)', weight='bold')
plt.show()


# SECCIÓN 5: ENTRENAMIENTO DEL MODELO DE REGRESIÓN LINEAL (OLS)

Separaremos el conjunto de datos en una matriz de diseño paramétrico ($X$, 2D) y un vector unidimensional de respuestas ($y$, 1D) para realizar el ajuste OLS iterativo.


In [ ]:
X_feature = df[['Horas_Estudio']]
y_target = df['Calificacion']

# Ajuste del modelo
modelo = LinearRegression()
modelo.fit(X_feature, y_target)

# Extracción paramétrica
beta_0 = modelo.intercept_
beta_1 = modelo.coef_[0]

print(f"Ecuación Ajustada: Y = {beta_0:.4f} + {beta_1:.4f}X")

plt.figure(figsize=(9, 6))
sns.scatterplot(x=df['Horas_Estudio'], y=df['Calificacion'], color='#38BDF8', edgecolor='#003865')

x_rango = np.linspace(df['Horas_Estudio'].min(), df['Horas_Estudio'].max(), 100)
y_predicha = beta_0 + beta_1 * x_rango
plt.plot(x_rango, y_predicha, color='#E85C0B', linewidth=3, label='Regresión OLS Ajustada')
plt.legend()
plt.title('Regresión Lineal Simple: Curva de Estimación Óptima', weight='bold')
plt.show()


# SECCIÓN 6: EVALUACIÓN DEL MODELO Y ANÁLISIS DE RESIDUOS

Un modelo ajustado requiere diagnóstico estadístico de sus residuales ($e_i = y_i - \hat{y}_i$) para certificar la homogeneidad de la varianza (Homocedasticidad).


In [ ]:
y_estimada = modelo.predict(X_feature)
residuos = y_target - y_estimada

r_cuadrado = metrics.r2_score(y_target, y_estimada)
rmse = np.sqrt(metrics.mean_squared_error(y_target, y_estimada))

print(f"Coeficiente de Determinación (R^2): {r_cuadrado:.4f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.4f}")

plt.figure(figsize=(9, 5))
plt.scatter(y_estimada, residuos, color='#003865', alpha=0.6)
plt.axhline(y=0, color='#E85C0B', linestyle='--', linewidth=2)
plt.title('Diagnóstico de Homocedasticidad (Distribución de Residuos)', weight='bold')
plt.xlabel('Valores Estimados (\hat{Y})')
plt.ylabel('Magnitud del Residuo (e)')
plt.show()


# SECCIÓN 7: DISCUSIÓN CRÍTICA Y RETOS GRADUADOS (EVALUACIÓN BLOOM)

El análisis paramétrico no sustituye la intuición y el pensamiento crítico. Proceda a resolver analíticamente los siguientes problemas, y posteriormente ingrese al Simulador Interactivo para poner a prueba su heurística visual.

**Acceso al Simulador:** [game_semana_6.html](./game_semana_6.html) (Nivel 7 evalúa directamente el impacto de los Outliers que calcularemos a continuación).

### Reto 1 (Aplicar)
Utilizando el objeto `modelo` ajustado, prediga la calificación esperada para un estudiante que dedica $18.5$ horas de estudio semanales.


In [ ]:
# Escriba su solución aquí (Reto 1)



### Reto 2 (Analizar)
Filtre el DataFrame `df` para eliminar los 3 outliers inyectados intencionalmente al final del dataset (observaciones donde $X > 27$ e $Y < 20$). Recalcule la correlación de Pearson y ajuste un nuevo modelo lineal. Imprima cómo varió la pendiente $\beta_1$ y el $R^2$ sin la presencia de estos valores anómalos.


In [ ]:
# Escriba su solución aquí (Reto 2)



### Reto 3 (Evaluar)
Construya un vector unidimensional $X$ distribuido uniformemente entre $[-10, 10]$. Genere $Y$ utilizando la ecuación no lineal estricta $Y = X^2$. Imprima el coeficiente de correlación de Pearson. Justifique matemáticamente y teóricamente el resultado mediante un comentario.


In [ ]:
# Escriba su solución aquí (Reto 3)

